In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

# Memuat data yang telah di-preprocessing
df = pd.read_csv('/content/drive/MyDrive/Muhammad Yahya Ayyasy_Tugas 21 Agustus 2026/data_preprocessed.csv')
df = df.dropna(subset=['teks_processed', 'sentimen'])

X = df['teks_processed']
y = df['sentimen']

# Split SEBELUM vektorisasi (Sesuai instruksi jobsheet untuk mencegah data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=500)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print('Shape train:', X_train_tfidf.shape)
print('Shape test :', X_test_tfidf.shape)

Shape train: (6358, 500)
Shape test : (1590, 500)


In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

pred_nb = nb_model.predict(X_test_tfidf)

print('\n=== NAIVE BAYES ===')
print(f'Akurasi: {accuracy_score(y_test, pred_nb):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred_nb))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, pred_nb))


=== NAIVE BAYES ===
Akurasi: 0.8931

Classification Report:
              precision    recall  f1-score   support

     negatif       0.87      0.98      0.92      1056
     positif       0.96      0.71      0.82       534

    accuracy                           0.89      1590
   macro avg       0.91      0.85      0.87      1590
weighted avg       0.90      0.89      0.89      1590


Confusion Matrix:
[[1039   17]
 [ 153  381]]


In [ ]:
# 2. MODEL SVM (LinearSVC)
# (Penambahan parameter dual=False untuk menstabilkan konvergensi pada dataset besar)
svm_model = LinearSVC(C=1.0, random_state=42, max_iter=2000, dual=False)
svm_model.fit(X_train_tfidf, y_train)

pred_svm = svm_model.predict(X_test_tfidf)

print('\n=== SVM ===')
print(f'Akurasi: {accuracy_score(y_test, pred_svm):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, pred_svm))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, pred_svm))


=== SVM ===
Akurasi: 0.9333

Classification Report:
              precision    recall  f1-score   support

     negatif       0.94      0.96      0.95      1056
     positif       0.91      0.88      0.90       534

    accuracy                           0.93      1590
   macro avg       0.93      0.92      0.92      1590
weighted avg       0.93      0.93      0.93      1590


Confusion Matrix:
[[1012   44]
 [  62  472]]


In [ ]:
joblib.dump(svm_model, 'svm_sentiment.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

print('\nModel dan vectorizer berhasil disimpan.')


Model dan vectorizer berhasil disimpan.


In [ ]:
# ==========================================
# 4. PENGUJIAN REVIEW BARU (DIPERBAIKI + INSTALL)
# ==========================================
# Install ulang Sastrawi untuk mencegah ModuleNotFoundError jika runtime terputus
!pip install Sastrawi

import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Deklarasi ulang engine Sastrawi
stemmer = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_full(text):
    text = clean_text(text)
    text = stopword.remove(text)
    text = stemmer.stem(text)
    return text

# Teks disesuaikan ke konteks DANA agar relevan secara bisnis
review_baru = [
    "Aplikasi DANA sangat bagus, transfer cepat dan praktis",
    "Kecewa banget, top up saldo nyangkut dan cs lambat",
    "Lumayan lah, sangat membantu buat bayar QRIS"
]

# Terapkan fungsi preprocess_full yang sudah dideklarasikan ulang
review_clean = [preprocess_full(r) for r in review_baru]
review_vec   = tfidf.transform(review_clean)

print('\n--- HASIL PREDIKSI DATA BARU ---')
print('Prediksi Naive Bayes:', nb_model.predict(review_vec))
print('Prediksi SVM        :', svm_model.predict(review_vec))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.0 MB/s eta 0:00:00

--- HASIL PREDIKSI DATA BARU ---
Prediksi Naive Bayes: ['positif' 'negatif' 'positif']
Prediksi SVM        : ['positif' 'negatif' 'positif']
